# Les 10 - AI-agenten in productie

In deze les leer je **productiepatronen** voor AI-agenten met behulp van het Microsoft Agent Framework (Python). We behandelen:

- **Observeerbaarheid** — het toevoegen van timing en logging aan agentinteracties
- **Evaluatie** — het gebruik van een evaluator-agent om de kwaliteit van antwoorden te scoren
- **Kostenbeheer** — strategieën voor tokenoptimalisatie en modelselectie

Het scenario is een **reisagent** die gebruikers helpt reizen te plannen, met monitoring en evaluatie daarbovenop.


## Installatie


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Overwegingen voor productie

Het verplaatsen van AI-agenten van prototypes naar productie vereist zorgvuldige aandacht voor drie pijlers:

1. **Observeerbaarheid** — Je hebt zicht nodig op wat de agent doet, hoe lang het duurt, en welke tools hij aanroept. Zonder tracing en logging is het vrijwel onmogelijk om productieproblemen te debuggen.

2. **Evaluatie** — Geautomatiseerde kwaliteitscontroles zorgen ervoor dat de reacties van de agent in de loop van de tijd nauwkeurig, volledig en nuttig blijven. Een evaluatie-agent kan reacties scoren aan de hand van gedefinieerde criteria.

3. **Kostenbeheer** — Tokengebruik heeft directe invloed op de kosten. Strategieën zoals promptoptimalisatie, modelkeuze en caching helpen de uitgaven onder controle te houden zonder in te boeten op kwaliteit.


## Een observeerbare agent bouwen

We definiëren reishulpmiddelen en wikkelen de agent-aanroep in met timing zodat we de latentie kunnen monitoren. In productie zou je integreren met OpenTelemetry of een vergelijkbare tracing-backend.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Evaluatiepatronen

Een veelvoorkomend productiepatroon is om een tweede agent te gebruiken als een **beoordelaar**. De beoordelaar beoordeelt de reactie van de primaire agent aan de hand van vooraf gedefinieerde criteria zoals volledigheid, nauwkeurigheid en behulpzaamheid.

Dit maakt het mogelijk:
- Geautomatiseerde kwaliteitscontroles voordat antwoorden gebruikers bereiken
- Detectie van regressies wanneer prompts of modellen veranderen
- Doorlopende monitoring van de prestaties van de agent in de loop van de tijd


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Kostenbeheersingsstrategieën

Het beheersen van kosten is cruciaal voor productie-AI-agents. Hier zijn belangrijke strategieën:

| Strategie | Beschrijving |
|---|---|
| **Promptoptimalisatie** | Houd systeeminstructies beknopt. Verwijder overbodige context om het aantal invoertokens te verminderen. |
| **Modelselectie** | Gebruik kleinere, goedkopere modellen (bijv. GPT-4o-mini) voor eenvoudige taken zoals classificatie of extractie, en reserveer grotere modellen voor complexe redenering. |
| **Caching** | Sla toolresultaten en veelvoorkomende queries in de cache op om redundante API-aanroepen te vermijden. |
| **Tokenbudgetten** | Stel `max_tokens`-limieten in om onverwacht lange antwoorden te voorkomen. |
| **Batchverwerking** | Groepeer meerdere gebruikersvragen in één enkele API-aanroep waar mogelijk. |

In de praktijk werkt een gelaagde aanpak goed: stuur eenvoudige verzoeken naar een snel, goedkoop model en schakel alleen voor complexe vragen over naar een krachtiger model.


## Samenvatting

In deze les heb je geleerd hoe je:

1. **Observability toevoegen** aan agentinteracties met timing en logging, waarmee de basis wordt gelegd voor tracing en monitoring.
2. **Agentreacties automatisch evalueren** met een evaluator-agent die volledigheid, nauwkeurigheid en behulpzaamheid beoordeelt.
3. **Kosten beheren** door promptoptimalisatie, modelselectie, caching en tokenbudgetten.

Deze productiepatronen helpen ervoor te zorgen dat je AI-agents betrouwbaar, meetbaar, en kosteneffectief zijn op schaal.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Disclaimer:
Dit document is vertaald met behulp van de AI-vertalingsdienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, houd er rekening mee dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het oorspronkelijke document in de oorspronkelijke taal geldt als de gezaghebbende bron. Voor cruciale informatie raden wij een professionele menselijke vertaling aan. Wij zijn niet aansprakelijk voor misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
